<a href="https://colab.research.google.com/github/brendalauriane/Docker-jenkins/blob/main/Projet1_Estimation_Appartements_Paris.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projet 1 — Estimer le prix d'un appartement à Paris
**Machine Learning · Régression linéaire · projet individuel**
Agence : Clés de Lutèce (fictive) · Données : DGFiP, ventes d'appartements à Paris en 2025

> **Mode d'emploi.** Chaque étape a un titre, son code, et **une ou deux phrases** qui disent *ce que j'ai constaté* et *ce que j'ai décidé*. Les blocs « Questions » sont là pour te guider : réponds-y dans les cellules « Constat / décision », puis supprime les consignes si tu veux un notebook plus propre.
> Avant de rendre : *Exécution › Tout exécuter* sans erreur, puis *Partager › Accès général › Tous les utilisateurs disposant du lien*.

## 0. Mise en place
Importer les bibliothèques et charger le fichier `ventes_appartements_paris_2025.zip`.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

SEED = 42
df_brut = pd.read_csv("D:/Downloads/ventes_appartements_paris_2025.zip")
df_brut.head()

FileNotFoundError: [Errno 2] No such file or directory: 'D:/Downloads/ventes_appartements_paris_2025.zip'

## 1. Inspecter les données
**Questions**
- Combien de lignes et de colonnes ? Quels types ?
- Où sont les valeurs manquantes (et combien) ?
- Quelles valeurs extrêmes ou incohérentes vois-tu (surface 0 ou énorme, prix très bas ou très haut, surface Carrez > surface habitable…) ?

**Livrable : au moins 3 problèmes notés.**

In [ ]:
# TODO : shape, info(), describe(), valeurs manquantes (isna().sum())

**Constat / décision :**
1. Problème 1 : …
2. Problème 2 : …
3. Problème 3 : …

## 2. Doublons
**Questions**
- Y a-t-il des lignes entièrement identiques ? Des ventes avec le même `id_vente` ?
- Une ligne répétée est-elle une erreur, ou plusieurs vraies ventes très semblables ? Que décides-tu, et pourquoi ?

In [ ]:
# TODO : df.duplicated().sum(), doublons sur id_vente, etc.

**Constat / décision :** …

## 3. Périmètre : garder les « appartements ordinaires »
**Questions** (chaque règle = une phrase de justification)
- Quelles ventes ne sont pas celles d'un appartement ordinaire ? Pense à : surfaces minuscules ou immenses, prix au m² aberrants, 0 pièce ou beaucoup de pièces, nombre de lots élevé (vente en bloc ?), prix symboliques…
- Combien de lignes écartes-tu avec chaque règle ? Combien en reste-t-il ?
- Comment traites-tu les valeurs manquantes ici : tu les gardes (elles seront imputées dans le pipeline) ou tu supprimes ? Attention : l'appartement B n'a pas de nombre de pièces.

In [ ]:
df = df_brut.copy()
# TODO : appliquer les règles une par une, en affichant le nombre de lignes écartées
# ex : df["prix_m2"] = df["prix_eur"] / df["surface_m2"]  (variable de travail, à ne PAS donner au modèle)
print(len(df_brut), "->", len(df))

**Règles retenues :**
- Règle 1 : … (parce que …)
- Règle 2 : … (parce que …)

## 4. Explorer (2 ou 3 graphiques)
Chaque graphique est **suivi de ce qu'il montre**.
Idées : prix vs surface ; prix au m² par arrondissement (boxplot) ; distribution du prix (est-elle très asymétrique ?).

In [ ]:
# TODO : graphique 1

**Ce que montre le graphique 1 :** …

In [ ]:
# TODO : graphique 2

**Ce que montre le graphique 2 :** …

In [ ]:
# TODO (optionnel) : graphique 3

**Ce que montre le graphique 3 :** …

## 5. Séparer : 20 % de test mis de côté
**Règle d'or :** le découpage se fait **avant** tout calcul appris (imputation, normalisation, encodage). Le jeu de test ne sert qu'**une seule fois**, à la toute fin, sur la version retenue.

**Questions**
- Quelle est la cible `y` ? Quelles colonnes mets-tu dans `X` ? (`id_vente` ne sert pas à prédire.)
- Pourquoi fixer `random_state` ?

In [ ]:
# TODO : définir X et y, puis train_test_split(test_size=0.2, random_state=SEED)
# X_train, X_test, y_train, y_test = ...

## 6. Référence : une règle simple, sans modèle
**Questions**
- Quelle règle simple un agent pourrait-il utiliser ? (ex : prix médian au m² × surface ; ou médiane par arrondissement.)
- Calcule-la **sur le train uniquement**, puis mesure son erreur moyenne **en euros** (MAE) sur des données qu'elle n'a pas vues.
- Ton modèle devra faire mieux (ou tu devras dire honnêtement qu'il ne fait pas mieux).

In [ ]:
# TODO : règle de référence + MAE en euros

**Constat :** la référence se trompe de … € en moyenne.

## 7. Pipeline scikit-learn
**Questions**
- Quelles colonnes sont de **vrais nombres** (surface, pièces, dépendances…) ? Lesquelles sont des **catégories écrites avec des chiffres** (arrondissement, mois…) ?
- Traitement des nombres : imputation (médiane) + normalisation. Traitement des catégories : encodage one-hot.
- Tout est réuni dans un `Pipeline` : imputation + encodage + normalisation + `LinearRegression`.

In [ ]:
# TODO : colonnes numériques / catégorielles, ColumnTransformer, Pipeline
# Astuce : écris une fonction construire_pipeline(num_cols, cat_cols) pour réutiliser à l'étape 8.

## 8. Versions : au moins quatre, comparées par validation croisée
**Règle :** une seule chose change d'une version à l'autre (ex : V1 surface seule ; V2 + arrondissement ; V3 + pièces et dépendances ; V4 prix en logarithme ou autre ajout).

**Questions**
- Quelle version retiens-tu ? L'écart avec les autres est-il **réel** (compare-le à la variation d'un pli à l'autre de la validation croisée) ?
- Le jeu de test n'est **pas** utilisé ici : seulement `X_train`.

In [ ]:
# TODO : boucle sur les versions, cross_val_score sur X_train (scoring MAE en euros), tableau récapitulatif (DataFrame)

**Version retenue :** … **parce que** … L'écart avec la suivante est / n'est pas réel car …

### Mesure finale sur le test (une seule fois)
Entraîner la version retenue sur tout le train, puis mesurer la MAE sur le test. Comparer à la référence de l'étape 6.

In [ ]:
# TODO : fit sur train, MAE sur test, comparaison avec la référence

## 9. Interpréter
**Questions**
- Traduis **au moins 3 coefficients ou effets** en phrases (ex : « à arrondissement égal, 10 m² de plus ajoutent environ … € »).
- Examine les **plus grosses erreurs** : quels appartements ? Qu'ont-ils en commun ?
- Qu'est-ce que le modèle **ne sait pas** (étage, état, vue, rue, travaux, ascenseur…) ?

In [ ]:
# TODO : coefficients (noms via get_feature_names_out), top des plus grosses erreurs sur le test

**Interprétation :**
1. …
2. …
3. …

**Ce que le modèle ne sait pas :** …

## 10. Livrer
Enregistrer le meilleur pipeline dans un fichier, **le recharger**, puis estimer les appartements A et B.

| | Appartement A | Appartement B |
|---|---|---|
| Arrondissement | 11e | 7e |
| Surface | 45 m² | 80 m² |
| Pièces principales | 2 | non renseigné |
| Dépendances (cave, parking) | 1 | 0 |

Attention : les autres colonnes attendues par le modèle (mois, nb_lots, surface_carrez_m2…) doivent recevoir une valeur raisonnable ou `NaN` (si le pipeline les impute) — justifie ton choix.

In [ ]:
# TODO : joblib.dump(pipeline, "modele_prix_paris.joblib")
# TODO : modele = joblib.load("modele_prix_paris.joblib")
# TODO : DataFrame des appartements A et B (mêmes colonnes que X), puis modele.predict(...)

## Note de conclusion pour la direction
*Sans jargon, lisible par l'agence.* Elle doit contenir :

1. **La question** posée par l'agence.
2. **Les données** gardées et écartées (et pourquoi, en une phrase chacune).
3. **L'erreur moyenne en euros**, comparée à la règle simple.
4. **Les estimations** des appartements A et B.
5. **Votre recommandation** : peut-on mettre l'outil en ligne ? Sous quelle forme (chiffre unique, fourchette, pas d'estimation pour certains cas…) ? Rappel : l'agence préfère ne rien afficher plutôt qu'un chiffre faux.
6. **Les limites** : pour quels appartements ne pas lui faire confiance ?

**Rédige ici :**

…